In [ ]:
import stumpy
import pandas as pd
import numpy as np
import os
import time

In [ ]:
%matplotlib inline


import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib import animation
from IPython.display import HTML

In [ ]:
def value_score_plot(_values, _scores, _title_values, _title_scores):
    fig, ax = plt.subplots(2, sharex=True, gridspec_kw={"hspace": 0.2})
    _values.plot(figsize=(15, 6), title=_title_values, ax=ax[0])
    _scores.plot(title=_title_scores, ax=ax[1], c="orange")

In [ ]:
def process_detector_frame(df):
    data = df.copy()
    data["datetime"] = pd.to_datetime(data["datetime"])
    det_names = [
        "walter",
        "median_absolute_deviation",
        "first_hour_average",
        "stddev_from_average",
        "mean_subtraction_cumulation",
        "stddev_from_moving_average",
        "least_squares",
        "histogram_bins",
    ]
    data["score"] = data[det_names].sum(axis=1) / len(det_names)
    return data[["timestamp", "value", "score", "datetime"]]

## MySQL

In [ ]:
file_path = "data/mysql#performance#events_statements#max#timer#wait-no_subject.csv"
mysql = pd.read_csv(file_path)
data = process_detector_frame(mysql)

In [ ]:
data.head()

In [ ]:
value_score_plot(
    _values=data.set_index("datetime")["value"],
    _scores=data.set_index("datetime")["score"],
    _title_values=file_path.replace("data/", "").replace(".csv", ""),
    _title_scores="Anomaly score",
)

In [ ]:
T_full = data.value.values.astype(float).copy()
m = int(60 * 24 * 2)
train_fraction = max(60 * 3, m + 1)
T_stream = T_full[:train_fraction]

stream = stumpy.stumpi(T_stream, m, egress=False)

windows = [(stream.P_, T_stream)]
P_max = -1

# Incrementally add one new data point at a time and update the matrix profile
for i in range(len(T_stream), len(T_full)):
    t = T_full[i]
    stream.update(t)

    if i % 50 == 0:
        windows.append((stream.P_, T_full[: i + 1]))
        if stream.P_.max() > P_max and stream.P_.max() != np.inf:
            P_max = stream.P_.max()
            break

In [ ]:
stream.

In [ ]:
fig, axs = plt.subplots(2, sharex=True, gridspec_kw={"hspace": 0.1}, figsize=(15, 6))

# rect = Rectangle((643, 0), m, 40, facecolor="lightgrey")
# axs[0].add_patch(rect)
# rect = Rectangle((8724, 0), m, 40, facecolor="lightgrey")
# axs[0].add_patch(rect)
axs[0].set_xlim((0, T_full.shape[0]))
y_min_limit = 0.8 if T_full.min() > 0 else 1.1
y_max_limit = 1.1 if T_full.max() > 0 else 0.8
axs[0].set_ylim((y_min_limit * T_full.min(), y_max_limit * T_full.max()))
axs[1].set_xlim((0, T_full.shape[0]))
axs[1].set_ylim((-0.1, 1.1 * P_max))
# axs[0].axvline(x=643, linestyle="dashed")
# axs[0].axvline(x=8724, linestyle="dashed")
# axs[1].axvline(x=643, linestyle="dashed")
# axs[1].axvline(x=8724, linestyle="dashed")
axs[0].set_ylabel("Stream Flow", fontsize="10")
axs[1].set_ylabel("Matrix Profile", fontsize="10")
axs[1].set_xlabel("Time", fontsize="10")


lines = []
for ax in axs:
    (line,) = ax.plot([], [], lw=2)
    lines.append(line)
(line,) = axs[1].plot([], [], lw=2)
lines.append(line)


def init():
    for line in lines:
        line.set_data([], [])
    return lines


def animate(window):
    P, T = window
    for line, data in zip(lines, [T, P]):
        line.set_data(np.arange(data.shape[0]), data)

    return lines


anim = animation.FuncAnimation(
    fig, animate, init_func=init, frames=windows, interval=100, blit=True, repeat=False
)

anim_out = anim.to_jshtml()
plt.tight_layout()
plt.close()  # Prevents duplicate image from displaying
if os.path.exists("None0000000.png"):
    os.remove("None0000000.png")  # Delete rogue temp file

HTML(anim_out)
# anim.save('/tmp/stumpi.mp4')

In [ ]:
anim.save('video/mysql_4_2.mp4')

## Log frequency

In [ ]:
logfreq.columns

In [ ]:
logfreq = pd.read_csv("data/redissite_logfrequency.csv")
logfreq["datetime"] = pd.to_datetime(logfreq["datetime"])

In [ ]:
logfreq.set_index("datetime")["value"].plot(figsize=(15, 6), title="redissite_logfrequency")

In [ ]:
logfreq.describe()

In [ ]:
value_score_plot(
    _values=logfreq.set_index("datetime")["value"],
    _scores=logfreq.set_index("datetime")["score"],
    _title_values="redissite_logfrequency values",
    _title_scores="Anomaly score",
)

## Matrix Profile

In [ ]:
T_full = logfreq.value.values.astype(float).copy()[: 60 * 24 * 7 * 2]
T_stream = T_full[: 60 * 3]
m = 10

stream = stumpy.stumpi(T_stream, m, egress=False)

windows = [(stream.P_, T_stream)]
P_max = -1

# Incrementally add one new data point at a time and update the matrix profile
for i in range(len(T_stream), len(T_full)):
    t = T_full[i]
    stream.update(t)

    if i % 50 == 0:
        windows.append((stream.P_, T_full[: i + 1]))
        if stream.P_.max() > P_max:
            P_max = stream.P_.max()

In [ ]:
P_max

In [ ]:
fig, axs = plt.subplots(2, sharex=True, gridspec_kw={"hspace": 0.1}, figsize=(15, 6))

# rect = Rectangle((643, 0), m, 40, facecolor="lightgrey")
# axs[0].add_patch(rect)
# rect = Rectangle((8724, 0), m, 40, facecolor="lightgrey")
# axs[0].add_patch(rect)
axs[0].set_xlim((0, T_full.shape[0]))
y_min_limit = 0.8 if T_full.min() > 0 else 1.1
y_max_limit = 1.1 if T_full.max() > 0 else 0.8
axs[0].set_ylim((y_min_limit * T_full.min(), y_max_limit * T_full.max()))
axs[1].set_xlim((0, T_full.shape[0]))
axs[1].set_ylim((-0.1, 1.1 * P_max))
# axs[0].axvline(x=643, linestyle="dashed")
# axs[0].axvline(x=8724, linestyle="dashed")
# axs[1].axvline(x=643, linestyle="dashed")
# axs[1].axvline(x=8724, linestyle="dashed")
axs[0].set_ylabel("Stream Flow", fontsize="10")
axs[1].set_ylabel("Matrix Profile", fontsize="10")
axs[1].set_xlabel("Time", fontsize="10")


lines = []
for ax in axs:
    (line,) = ax.plot([], [], lw=2)
    lines.append(line)
(line,) = axs[1].plot([], [], lw=2)
lines.append(line)


def init():
    for line in lines:
        line.set_data([], [])
    return lines


def animate(window):
    P, T = window
    for line, data in zip(lines, [T, P]):
        line.set_data(np.arange(data.shape[0]), data)

    return lines


anim = animation.FuncAnimation(
    fig, animate, init_func=init, frames=windows, interval=100, blit=True, repeat=False
)

anim_out = anim.to_jshtml()
plt.tight_layout()
plt.close()  # Prevents duplicate image from displaying
if os.path.exists("None0000000.png"):
    os.remove("None0000000.png")  # Delete rogue temp file

HTML(anim_out)
# anim.save('/tmp/stumpi.mp4')

In [ ]:
anim.save('video/logfreq.mp4')

In [ ]:
! open .

## Different Window Sizes

In [ ]:
file_path = "data/mysql#performance#events_statements#count#star-no_subject.csv" 
# data = pd.read_csv(file_path, parse_dates=["datetime"])
# data["value"] = data["value"].astype(float)
mysql = pd.read_csv(file_path)
data = process_detector_frame(mysql)

In [ ]:
days_dict = {
    "10 minutes": 10,
    "30 minutes": 30,
    "1 hour": 60,
    "6 hours": 6 * 60,
    "12 hours": 12 * 60,
    "1 day": 24 * 60,
    "2 days": 2 * 24 * 60,
}

days_df = pd.DataFrame.from_dict(days_dict, orient="index", columns=["m"])
days_df

In [ ]:
data.dtypes

In [ ]:
data.head()

In [ ]:
n_points = 300000
x_axis_labels = data.iloc[np.arange(0, data[:n_points].shape[0], 24*60)]["datetime"].dt.strftime("%B %d").values
duration_dict = {v:k for k, v in days_df.to_dict()["m"].items()}

fig, axs = plt.subplots(
    days_df.shape[0], sharex=True, gridspec_kw={"hspace": 1}, figsize=(15, 10)
)
fig.text(0.5, -0.1, "Subsequence Start Date", ha="center", fontsize="10")
fig.text(0.08, 0.5, "Matrix Profile", va="center", rotation="vertical", fontsize="10")
for i, varying_m in enumerate(days_df["m"].values):
    mp = stumpy.stump(data[:n_points]["value"], varying_m)
    plot_vals = [0] * varying_m
    plot_vals.extend(list(mp[:, 0]))
    axs[i].plot(plot_vals)
#     axs[i].plot(mp[:, 0])
#     axs[i].set_ylim(0, 40)
#     axs[i].set_xlim(0, 3600)
    title = f"m = {duration_dict[varying_m]}"
    axs[i].set_title(title)
plt.xticks(np.arange(0, data[:n_points].shape[0], 24*60), x_axis_labels)
plt.xticks(rotation=75)
plt.suptitle("STUMP with Varying Window Sizes")
plt.show()

In [ ]:
value_score_plot(
    _values=data.set_index("datetime")["value"],
    _scores=data.set_index("datetime")["score"],
    _title_values=file_path.replace("data/", "").replace(".csv", ""),
    _title_scores="Anomaly score",
)

In [ ]:
data

## Anomaly detection based on top discords

In [ ]:
from tqdm import tqdm_notebook

### MP score

In [ ]:
def is_top_k_discord_score(arr, k=3):
    sorted_unique_discords = sorted(set(arr), reverse=True)[:k]
    if len(sorted_unique_discords) >= k:
        score = (
            arr[-1] / sorted_unique_discords[0]
            if arr[-1] < sorted_unique_discords[-1]
            else 1.0
        )
        return arr[-1] > sorted_unique_discords[-1], score
    elif len(sorted_unique_discords) == 1:
        return False, 0.0
    elif sorted_unique_discords:
        score = (
            arr[-1] / sorted_unique_discords[0]
            if arr[-1] < sorted_unique_discords[0]
            else 1.0
        )
        return arr[-1] > sorted_unique_discords[0], score
    else:
        return False, 0.0

In [ ]:
def mp_score_plot(data, threshold, mp_weight):
    _data = data.copy()
    _data["avg_score"] = (1 - mp_weight) * data["score"] + mp_weight * data["mp_score"]
    fig, ax = plt.subplots(
        4, sharex=True, gridspec_kw={"hspace": 0.4}, figsize=(15, 12)
    )
    _data.set_index("datetime")["value"].plot(
        title=f"MAD anomalies (threshold = {threshold})", ax=ax[0], alpha=0.8,
    )
    _data[_data.score > threshold].plot.scatter(
        x="datetime", y="value", ax=ax[0], color="red"
    )
    _data.set_index("datetime")["score"].plot(
        title="MAD anomaly score", ax=ax[1], c="orange"
    )
    _data.set_index("datetime")["value"].plot(
        title=f"MAD + MP anomalies (threshold = {threshold}, MAD weight = {1-mp_weight:.2f}, MP weight = {mp_weight:.2f})",
        ax=ax[2],
        alpha=0.8,
    )
    _data[_data.avg_score > threshold].plot.scatter(
        x="datetime", y="value", ax=ax[2], color="red"
    )
    _data.set_index("datetime")["avg_score"].plot(
        title="MAD + MP anomaly score", ax=ax[3], c="orange"
    )
    xc = _data.iloc[4 * 24 * 60]["datetime"]
    ax[2].axvline(x=xc, color="k", linestyle="--")
    ax[0].legend(["value", "IsAnomaly"])
    ax[2].legend(["value", "MP warmup", "IsAnomaly"])

In [ ]:
file_path = "data/docker#memory#usage#pct.csv"    
# data = pd.read_csv(file_path, parse_dates=["datetime"])
# data["value"] = data["value"].astype(float)
mysql = pd.read_csv(file_path)
data = process_detector_frame(mysql)

In [ ]:
value_score_plot(
    _values=data.set_index("datetime")["value"],
    _scores=data.set_index("datetime")["score"],
    _title_values=file_path.replace("data/", "").replace(".csv", ""),
    _title_scores="Anomaly score",
)

In [ ]:
# anomaly scores

T_full = data.value.values.astype(float).copy()
m = 24 * 60
train_fraction = max(60 * 3, m + 1)
T_stream = T_full[:train_fraction]

stream = stumpy.stumpi(T_stream, m, egress=False)
anomaly_labels = []
scores = []

# Incrementally add one new data point at a time and update the matrix profile
for i in tqdm_notebook(range(len(T_stream), len(T_full))):
    t = T_full[i]
    stream.update(t)
    label, score = is_top_k_discord_score(stream.P_)
    anomaly_labels.append(label)
    scores.append(score)

In [ ]:
labels = [False] * len(T_stream)
labels.extend(anomaly_labels)

mp_scores = [0.0] * len(T_stream)
mp_scores.extend(scores)

_data = data.copy()
_data["isAnomaly"] = labels
_data["mp_score"] = mp_scores

_data["avg_score"] = (_data["score"] + _data["mp_score"]) / 2

In [ ]:
_data.head()

In [ ]:
mp_score_plot(_data, threshold=0.75, mp_weight=1)

### Top 3 discords logic

In [ ]:
def is_top_k_discord(arr, k=3):
    sorted_unique_discords = sorted(set(arr), reverse=True)[:k]
    if len(sorted_unique_discords) >= k:
        return arr[-1] > sorted_unique_discords[-1]
    elif sorted_unique_discords:
        return arr[-1] > sorted_unique_discords[0]
    else:
        return False

In [ ]:
T_full = data.value.values.astype(float).copy()
m = 24 * 60
train_fraction = max(60 * 3, m + 1)
T_stream = T_full[:train_fraction]

stream = stumpy.stumpi(T_stream, m, egress=False)
anomaly_labels = []
scores = []

# Incrementally add one new data point at a time and update the matrix profile
for i in tqdm_notebook(range(len(T_stream), len(T_full))):
    t = T_full[i]
    stream.update(t)
    anomaly_labels.append(is_top_k_discord(stream.P_))

In [ ]:
labels = [False] * len(T_stream)
labels.extend(anomaly_labels)

In [ ]:
len(labels)

In [ ]:
_data = data.copy()
_data["isAnomaly"] = labels

In [ ]:
_data[_data.isAnomaly]

In [ ]:
ax = data.set_index("datetime")[["value"]].plot(
    figsize=(15, 6), alpha=0.6, 
    title=f"1 day sliding window | top 3 discords",
#     title=f"{m} minutes sliding window | top 3 discords"
)
_data[_data.isAnomaly].plot.scatter(x="datetime", y="value", ax=ax, color="red")
ax.legend(["value", "IsAnomaly"])

## Annotation vector

In [ ]:
file_path = "data/mysql#status#handler#read#first-no_subject.csv"   
mysql = pd.read_csv(file_path)
data = process_detector_frame(mysql)

data["weekday"] = data.datetime.dt.weekday

AV = data.weekday.values.copy()
AV[AV < 5] = 0
AV[AV >= 5] = 1

updated_values = data.value.values.copy()
updated_values[np.where(AV != 0)] = 0

data["value"] = updated_values

In [ ]:
data

In [ ]:
value_score_plot(
    _values=data.set_index("datetime")["value"],
    _scores=data.set_index("datetime")["score"],
    _title_values=file_path.replace("data/", "").replace(".csv", ""),
    _title_scores="Anomaly score",
)

In [ ]:
T_full = data.value.values.astype(float).copy()
m = 24 * 60
train_fraction = max(60 * 3, m + 1)
T_stream = T_full[:train_fraction]

stream = stumpy.stumpi(T_stream, m, egress=False)
anomaly_labels = []
scores = []

# Incrementally add one new data point at a time and update the matrix profile
for i in tqdm_notebook(range(len(T_stream), len(T_full))):
    t = T_full[i]
    stream.update(t)
    anomaly_labels.append(is_top_k_discord(stream.P_))

labels = [False] * len(T_stream)
labels.extend(anomaly_labels)

In [ ]:
_data = data.copy()
_data["isAnomaly"] = labels

In [ ]:
ax = data.set_index("datetime")[["value"]].plot(
    figsize=(15, 6), alpha=0.6, 
    title=f"1 day sliding window | top 3 discords",
#     title=f"{m} minutes sliding window | top 3 discords"
)
_data[_data.isAnomaly].plot.scatter(x="datetime", y="value", ax=ax, color="red")
ax.legend(["value", "IsAnomaly"])

In [ ]:
n_points = 300000
x_axis_labels = (
    data.iloc[np.arange(0, data[:n_points].shape[0], 24 * 60)]["datetime"]
    .dt.strftime("%B %d")
    .values
)
duration_dict = {v: k for k, v in days_df.to_dict()["m"].items()}

fig, axs = plt.subplots(
    days_df.shape[0] * 2, sharex=True, gridspec_kw={"hspace": 1}, figsize=(15, 15)
)
fig.text(0.5, -0.1, "Subsequence Start Date", ha="center", fontsize="10")
fig.text(0.08, 0.5, "Matrix Profile", va="center", rotation="vertical", fontsize="10")
for i, varying_m in enumerate(days_df["m"].values):
    mp = stumpy.stump(data[:n_points]["value"], varying_m)
    plot_vals = [0] * (varying_m - 1)
    plot_vals.extend(list(mp[:, 0]))
    updated_mp = plot_vals + (1 - AV[:n_points]) * max(plot_vals)
    axs[2 * i].plot(plot_vals)
    title = f"m = {duration_dict[varying_m]}"
    axs[2 * i].set_title(title)
    axs[2 * i + 1].plot(updated_mp)
    title = f"m = {duration_dict[varying_m]} (corrected MP)"
    axs[2 * i + 1].set_title(title)
plt.xticks(np.arange(0, data[:n_points].shape[0], 24 * 60), x_axis_labels)
plt.xticks(rotation=75)
plt.suptitle("STUMP with Varying Window Sizes")
plt.show()

## Multidimensional MP

In [ ]:
df_names = [
    "data/mysql#status#handler#read#first-no_subject.csv",
    "data/mysql#performance#events_statements#max#timer#wait-no_subject.csv",
    "data/mysql#performance#events_statements#count#star-no_subject.csv",
    "data/mysql#performance#table_io_waits#count#fetch-no_subject.csv",
]

In [ ]:
d = {}
for i, name in enumerate(df_names):
    d[f"T{i+1}"] = pd.read_csv(name)["value"].values

In [ ]:
df = pd.DataFrame(d)

In [ ]:
fig, axs = plt.subplots(
    df.shape[1], sharex=True, gridspec_kw={"hspace": 0}, figsize=(15, 8)
)
plt.suptitle("Multi-dimensional TS")

for i in range(df.shape[1]):
    axs[i].set_ylabel(f"T{i + 1}")
    axs[i].set_xlabel("Time")
    axs[i].plot(df[f"T{i + 1}"])

plt.show()

In [ ]:
m = 60 * 24
start_t_mstump = time.time()
mps, indices = stumpy.mstump(df, m, discords=True)
end_t_mstump = time.time()
motifs_idx = np.argsort(mps, axis=1)[:, :2]
k = 2
start_t_subspace = time.time()
S = stumpy.subspace(
    df, m, motifs_idx[k][0], indices[k][motifs_idx[k][0]], k, discords=True
)
end_t_subspace = time.time()

In [ ]:
print(f"Data shape: {df.shape}, sliding window - {m}, time mstump: {end_t_mstump - start_t_mstump}")

In [ ]:
mps.shape[1] + m, df.shape

In [ ]:
mps.shape

In [ ]:
fig, axs = plt.subplots(
    mps.shape[0] * 2, sharex=True, gridspec_kw={"hspace": 0}, figsize=(15, 8)
)

for k, dim_name in enumerate(df.columns):
    axs[k].set_ylabel(dim_name)
    axs[k].plot(df[dim_name])
    axs[k].set_xlabel("Time")

    axs[k + mps.shape[0]].set_ylabel(dim_name.replace("T", "P"))
    plot_vals = [0] * m
    plot_vals.extend(list(mps[k]))
    axs[k + mps.shape[0]].plot(plot_vals, c="orange")
    #     axs[k + mps.shape[0]].plot(mps[k], c="orange")
    axs[k + mps.shape[0]].set_xlabel("Time")

#     axs[k].axvline(x=motifs_idx[1, 0], linestyle="dashed", c="black")
#     axs[k].axvline(x=motifs_idx[1, 1], linestyle="dashed", c="black")
#     axs[k + mps.shape[0]].axvline(x=motifs_idx[1, 0], linestyle="dashed", c="black")
#     axs[k + mps.shape[0]].axvline(x=motifs_idx[1, 1], linestyle="dashed", c="black")

#     if dim_name != "T3":
#         axs[k].plot(
#             range(motifs_idx[k, 0], motifs_idx[k, 0] + m),
#             df[dim_name].iloc[motifs_idx[k, 0] : motifs_idx[k, 0] + m],
#             c="red",
#             linewidth=4,
#         )
#         axs[k].plot(
#             range(motifs_idx[k, 1], motifs_idx[k, 1] + m),
#             df[dim_name].iloc[motifs_idx[k, 1] : motifs_idx[k, 1] + m],
#             c="red",
#             linewidth=4,
#         )
#         axs[k + mps.shape[0]].plot(
#             motifs_idx[k, 0],
#             mps[k, motifs_idx[k, 0]] + 1,
#             marker="v",
#             markersize=10,
#             color="red",
#         )
#         axs[k + mps.shape[0]].plot(
#             motifs_idx[k, 1],
#             mps[k, motifs_idx[k, 1]] + 1,
#             marker="v",
#             markersize=10,
#             color="red",
#         )
#     else:
#         axs[k + mps.shape[0]].plot(
#             motifs_idx[k, 0],
#             mps[k, motifs_idx[k, 0]] + 1,
#             marker="v",
#             markersize=10,
#             color="black",
#         )
#         axs[k + mps.shape[0]].plot(
#             motifs_idx[k, 1],
#             mps[k, motifs_idx[k, 1]] + 1,
#             marker="v",
#             markersize=10,
#             color="black",
#         )
data = pd.read_csv(name)["datetime"]
data
x_axis_labels = (
    pd.read_csv(name, parse_dates=["datetime"])["datetime"].dt.strftime("%B %d").values
)
plt.xticks(np.arange(0, pd.read_csv(name).shape[0], 24 * 60), x_axis_labels[:: 24 * 60])
plt.xticks(rotation=75)
plt.show()

## Time of calculations

In [ ]:
from collections import defaultdict

In [ ]:
n = 2 * 24 * 60
# m = 24 * 60
ms = [10, 30, 60, 6 * 60, 12 * 60, 24 * 60, 2 * 24 * 60]
d = defaultdict(list)
for m in ms:
    idx = []
    print(m)
    for i in range(1, 16, 2):
        start_ts = time.time()
        mp = stumpy.stump(np.random.randn(int(i * n)), m)
        end_ts = time.time()
        d[m].append(end_ts - start_ts)
        if m == 10:
            d["num_point"].append(f"{int(i*n)} points ({2*i} days)")
        print(f"{int(i*n)} points ({2*i} days): {end_ts-start_ts:.3f} sec.")

In [ ]:
perf = pd.DataFrame(d)
perf[["num_point", 10, 30, 60, 360, 720, 1440, 2880]].to_csv("data/perf.csv")

In [ ]:
perf

In [ ]:
for col in perf.columns:
    if col != "num_point":
        perf[str(col) + " minutes"] = perf[col].apply(lambda x: int(60 / x))

In [ ]:
perf

In [ ]:
perf[
    [
        "num_point",
        "10 minutes",
        "30 minutes",
        "60 minutes",
        "360 minutes",
        "720 minutes",
        "1440 minutes",
        "2880 minutes",
    ]
].to_csv("data/perf.csv")